In [2]:
import pandas as pd

ada = pd.read_csv("csvs/position_pred_results_ADAUSDT.csv")
btc = pd.read_csv("csvs/position_pred_results_BTCUSDT.csv")
doge = pd.read_csv("csvs/position_pred_results_DOGEUSDT.csv")
eth = pd.read_csv("csvs/position_pred_results_ETHUSDT.csv")
ltc = pd.read_csv("csvs/position_pred_results_LTCUSDT.csv")
sol = pd.read_csv("csvs/position_pred_results_SOLUSDT.csv")
xrp = pd.read_csv("csvs/position_pred_results_XRPUSDT.csv")

ada["asset"] = "ADA"
btc["asset"] = "BTC"
doge["asset"] = "DOGE"
eth["asset"] = "ETH"
ltc["asset"] = "LTC"
sol["asset"] = "SOL"
xrp["asset"] = "XRP"

all_results = pd.concat(
    [ada, btc, doge, eth, ltc, sol, xrp],
    ignore_index=True
)

all_results["combined_bps"] = (all_results["mean_bps"] + all_results["mean_bps_ask"]) / 2

avg_all_results = (
    all_results
    .groupby(["strategy", "K", "weighting"])[["mean_bps", "mean_bps_ask", "combined_bps"]]
    .mean()
    .reset_index()
)

In [3]:
avg_all_results[
    (avg_all_results["strategy"] == "baseline") &
    (avg_all_results["K"] == 60) &
    (avg_all_results["weighting"] == "uniform") 
    ]

,strategy,K,weighting,mean_bps,mean_bps_ask,combined_bps
35,baseline,60,uniform,5.310507,4.435652,4.873079


In [25]:
avg_all_results[avg_all_results["strategy"] == "baseline"].sort_values("combined_bps").iloc[0, :]

strategy         baseline
K                      40
weighting       quadratic
mean_bps         4.898333
mean_bps_ask     4.046221
combined_bps     4.472277
Name: 22, dtype: object

In [18]:
avg_all_results.sort_values("combined_bps").head(10)

,strategy,K,weighting,mean_bps,mean_bps_ask,combined_bps
100,spread,25,uniform,3.257869,2.888373,3.073121
101,spread,30,uniform,3.264120,2.900211,3.082166
99,spread,20,uniform,3.311067,2.889124,3.100095
102,spread,35,uniform,3.289770,2.933484,3.111627
103,spread,40,uniform,3.353965,2.986301,3.170133
98,spread,15,uniform,3.418501,2.977421,3.197961
104,spread,45,uniform,3.410796,3.041712,3.226254
105,spread,50,uniform,3.472988,3.111509,3.292248
106,spread,55,uniform,3.541728,3.188651,3.365189
107,spread,60,uniform,3.612607,3.254431,3.433519


In [20]:
all_results.groupby("strategy")[["mean_bps", "mean_bps_ask", "combined_bps"]].mean().reset_index().sort_values("combined_bps")

,strategy,mean_bps,mean_bps_ask,combined_bps
6,spread,3.520889,3.144084,3.332487
3,mid,4.465276,3.815006,4.140141
2,micro,4.684582,3.899975,4.292278
7,vwap,4.741820,4.378423,4.560122
1,close,4.993883,4.126410,4.560146
0,baseline,5.143671,4.327245,4.735458
4,obi,5.684858,4.824433,5.254645
5,ofi,6.370415,5.252920,5.811668


# Best Metrics (at K = 60)

In [11]:
# filter to K = 60
k60 = all_results[all_results["K"] == 30]

# average across assets
k60_avg = (
    k60
    .groupby(["strategy", "weighting"])[["mean_bps", "mean_bps_ask", "combined_bps"]]
    .mean()
    .reset_index()
)

# rank strategies
k60_ranked = k60_avg.sort_values("combined_bps")

display(k60_ranked.head(15))

,strategy,weighting,mean_bps,mean_bps_ask,combined_bps
8,spread,uniform,3.264120,2.900211,3.082166
5,mid,uniform,4.316234,3.668795,3.992515
4,micro,uniform,4.536967,3.745940,4.141454
9,vwap,uniform,4.500038,4.165859,4.332949
0,baseline,linear,4.910893,4.053646,4.482269
3,close,uniform,4.955515,4.014122,4.484819
1,baseline,quadratic,4.910536,4.064138,4.487337
2,baseline,uniform,5.005430,4.145811,4.575620
6,obi,uniform,5.518696,4.579075,5.048885
7,ofi,uniform,6.108249,5.093979,5.601114


# Identify Best Weighting Strategy

In [5]:
# Baseline results averaged across crypto pairs
baseline = all_results[
    all_results["strategy"] == "baseline"
]

baseline_avg = (
    baseline
    .groupby(["K", "weighting"])[["mean_bps", "mean_bps_ask", "combined_bps"]]
    .mean()
    .reset_index()
    .sort_values(["combined_bps"])
)

display(baseline_avg)

,K,weighting,mean_bps,mean_bps_ask,combined_bps
22,40,quadratic,4.898333,4.046221,4.472277
19,35,quadratic,4.898924,4.049045,4.473985
25,45,quadratic,4.905585,4.052320,4.478952
15,30,linear,4.910893,4.053646,4.482269
12,25,linear,4.910877,4.054160,4.482519
16,30,quadratic,4.910536,4.064138,4.487337
28,50,quadratic,4.917256,4.062921,4.490089
18,35,linear,4.921798,4.068120,4.494959
9,20,linear,4.927131,4.076424,4.501778
31,55,quadratic,4.931479,4.075796,4.503637


# Best Weighting Strategy for 3 Best Metrics

In [6]:
from utils import PositionPredictor
from pathlib import Path
import numpy as np
import sys, os

ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']
OHLC_COLS = ['open', 'high', 'low', 'close']

price_cols = ASK_PRICE_COLS + BID_PRICE_COLS + OHLC_COLS
vol_cols = ASK_VOL_COLS + BID_VOL_COLS + ["volume"]

# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()

if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

cryptos = [f for f in os.listdir(root) if "USDT" in f]

y_trains = {}
vols = {}

for crypto in cryptos:

    symbol_dir = root / crypto
    X_path = symbol_dir / "X_train.parquet"
    X_test_path = symbol_dir / "X_test.parquet"
    Y_path = symbol_dir / "y_train.parquet"
    vol_path = symbol_dir / "vol_to_fill.txt"

    # X_train = pd.read_parquet(X_path).sort_values(["anonymized_id", "time_in_hour"])
    # X_test = pd.read_parquet(X_test_path).sort_values(["anonymized_id", "time_in_hour"])
    y_train = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])

    volume_to_fill = None
    if vol_path.exists():
        import re
        with open(vol_path) as f:
            m = re.search(r"([\d.]+)", f.read())
        if m:
            volume_to_fill = float(m.group(1))

    vols[crypto] = volume_to_fill


    ids = y_train["anonymized_id"].unique()
    times = np.sort(y_train["time_in_hour"].unique())

    full_index = pd.MultiIndex.from_product(
        [ids, times],
        names=["anonymized_id", "time_in_hour"]
    )

    y_train = (
        y_train
        .set_index(["anonymized_id", "time_in_hour"])
        .reindex(full_index)
        .reset_index()
    )

    # Volume fill with 0
    y_train[vol_cols] = y_train[vol_cols].fillna(0)

    y_train[price_cols] = (
        y_train
        .groupby("anonymized_id")[price_cols]
        .apply(lambda df: df.ffill().bfill())
        .reset_index(level=0, drop=True)
    )

    y_trains[crypto] = y_train

pp0 = PositionPredictor(y_trains[cryptos[0]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[0]])
pp1 = PositionPredictor(y_trains[cryptos[1]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[1]])
pp2 = PositionPredictor(y_trains[cryptos[2]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[2]])
pp3 = PositionPredictor(y_trains[cryptos[3]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[3]])
pp4 = PositionPredictor(y_trains[cryptos[4]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[4]])
pp5 = PositionPredictor(y_trains[cryptos[5]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[5]])
pp6 = PositionPredictor(y_trains[cryptos[6]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[6]])

pp0a = PositionPredictor(y_trains[cryptos[0]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[0]], side = "ask")
pp1a = PositionPredictor(y_trains[cryptos[1]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[1]], side = "ask")
pp2a = PositionPredictor(y_trains[cryptos[2]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[2]], side = "ask")
pp3a = PositionPredictor(y_trains[cryptos[3]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[3]], side = "ask")
pp4a = PositionPredictor(y_trains[cryptos[4]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[4]], side = "ask")
pp5a = PositionPredictor(y_trains[cryptos[5]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[5]], side = "ask")
pp6a = PositionPredictor(y_trains[cryptos[6]], ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS, vols[cryptos[6]], side = "ask")

pps = [pp0, pp1, pp2, pp3, pp4, pp5, pp6]
ppsa = [pp0a, pp1a, pp2a, pp3a, pp4a, pp5a, pp6a]


In [12]:
strategies = ["spread", "mid", "vwap"]
weightings = ["uniform", "linear", "quadratic"]
Ks = range(60, 1, -15)

rows = []

for strategy in strategies:
    for weighting in weightings:
        for K in Ks:

            bid_bps = []
            ask_bps = []

            for pp in pps:

                metrics = pp.compute_execution_metrics(
                    strategy=strategy,
                    weighting=weighting,
                    K_seconds=K
                )

                bid_bps.append(metrics["mean_bps"])

            for ppa in ppsa:

                metrics = ppa.compute_execution_metrics(
                    strategy=strategy,
                    weighting=weighting,
                    K_seconds=K
                )

                ask_bps.append(metrics["mean_bps"])

            rows.append({
                "strategy": strategy,
                "weighting": weighting,
                "K": K,
                "mean_bps": np.mean(bid_bps),
                "mean_bps_ask": np.mean(ask_bps),
                "combined_bps": (np.mean(bid_bps) + np.mean(ask_bps)) / 2
            })

results = pd.DataFrame(rows)

results = results.sort_values(["combined_bps"])

results.head()

,strategy,weighting,K,mean_bps,mean_bps_ask,combined_bps
6,spread,linear,30,3.220701,2.829769,3.025235
9,spread,quadratic,45,3.242622,2.857306,3.049964
5,spread,linear,45,3.237257,2.870319,3.053788
2,spread,uniform,30,3.264120,2.900211,3.082166
8,spread,quadratic,60,3.296062,2.933504,3.114783


In [13]:
# top 3 spread
"""
strategy	weighting	K	mean_bps	mean_bps_ask	combined_bps
	spread	linear	30	3.220701	2.829769	3.025235
	spread	quadratic	45	3.242622	2.857306	3.049964
	spread	linear	45	3.237257	2.870319	3.053788
"""
results[results["strategy"] == "spread"].iloc[:3,:]

,strategy,weighting,K,mean_bps,mean_bps_ask,combined_bps
6,spread,linear,30,3.220701,2.829769,3.025235
9,spread,quadratic,45,3.242622,2.857306,3.049964
5,spread,linear,45,3.237257,2.870319,3.053788


In [14]:
# top 3 mid
"""
	strategy	weighting	K	mean_bps	mean_bps_ask	combined_bps
	mid	quadratic	60	4.282471	3.622557	3.952514
	mid	linear	45	4.287504	3.633044	3.960274
	mid	linear	60	4.293517	3.630612	3.962065
"""
results[results["strategy"] == "mid"].iloc[:3,:]

,strategy,weighting,K,mean_bps,mean_bps_ask,combined_bps
20,mid,quadratic,60,4.282471,3.622557,3.952514
17,mid,linear,45,4.287504,3.633044,3.960274
16,mid,linear,60,4.293517,3.630612,3.962065


In [ ]:
# top 3 vwap
"""
	strategy	weighting	K	mean_bps	mean_bps_ask	combined_bps
	vwap	linear	60	4.235783	3.803718	4.019750
	vwap	quadratic	60	4.221470	3.841575	4.031523
	vwap	linear	45	4.300615	3.930006	4.115311
"""
results[results["strategy"] == "vwap"].iloc[:3,:]

,strategy,weighting,K,mean_bps,mean_bps_ask,combined_bps
28,vwap,linear,60,4.235783,3.803718,4.019750
32,vwap,quadratic,60,4.221470,3.841575,4.031523
29,vwap,linear,45,4.300615,3.930006,4.115311


# Combine 3 Best Metrics (see if this outperforms all other models)

The below strategies are informed by the previous section.

In [17]:
# let's literally get the positions for 

pos_spread = {}
pos_mid = {}
pos_v = {}

pos_spread_a = {}
pos_mid_a = {}
pos_v_a = {}

for i in range(7): # these are the best performing models for each metric
    pos_spread[i] = pps[i].compute_execution_metrics(strategy="spread", weighting="linear", K_seconds=30)["positions"]
    pos_mid[i] = pps[i].compute_execution_metrics(strategy="mid", weighting="quadratic", K_seconds=60)["positions"]
    pos_v[i] = pps[i].compute_execution_metrics(strategy="vwap", weighting="linear", K_seconds=60)["positions"]

    pos_spread_a[i] = ppsa[i].compute_execution_metrics(strategy="spread", weighting="linear", K_seconds=30)["positions"]
    pos_mid_a[i] = ppsa[i].compute_execution_metrics(strategy="mid", weighting="quadratic", K_seconds=60)["positions"]
    pos_v_a[i] = ppsa[i].compute_execution_metrics(strategy="vwap", weighting="linear", K_seconds=60)["positions"]

rows = []

for s in np.arange(7):
    for m in np.arange(7 - s):

        v = 6 - s - m

        total = s + m + v
        s_frac = s / total
        m_frac = m / total
        v_frac = v / total

        bids = []
        asks = []
        combineds = []

        for i in range(7):

            # generate positions
            positions = s_frac * pos_spread[i] + m_frac * pos_mid[i] + v_frac * pos_v[i]
            positions_a = s_frac * pos_spread_a[i] + m_frac * pos_mid_a[i] + v_frac * pos_v_a[i]

            # compute_execution_metrics with positions
            # bid
            bid_bps = pps[i].compute_execution_metrics(strategy="doesn't matter", positions=positions)["mean_bps"]
            # ask
            ask_bps = ppsa[i].compute_execution_metrics(strategy="doesn't matter", positions=positions_a)["mean_bps"]
            # combine
            combined_bps = (bid_bps + ask_bps) / 2

            bids.append(bid_bps)
            asks.append(ask_bps)
            combineds.append(combined_bps)
            
        # append
        rows.append({
            "s_frac": s_frac,
            "m_frac": m_frac,
            "v_frac": v_frac,
            "mean_bps": np.mean(bids),
            "mean_bps_ask": np.mean(asks),
            "combined_bps": np.mean(combineds),
        })

combinations_df = pd.DataFrame(rows)

combinations_df = combinations_df.sort_values("combined_bps")

combinations_df.to_csv("csvs/position_pred_results_spread_mid_volume.csv", index=False)

combinations_df

,s_frac,m_frac,v_frac,mean_bps,mean_bps_ask,combined_bps
25,0.833333,0.000000,0.166667,3.191609,2.796365,2.993987
27,1.000000,0.000000,0.000000,3.220701,2.829769,3.025235
22,0.666667,0.000000,0.333333,3.256261,2.856894,3.056577
23,0.666667,0.166667,0.166667,3.336393,2.899049,3.117721
26,0.833333,0.166667,0.000000,3.349615,2.920468,3.135042
19,0.500000,0.166667,0.333333,3.414158,2.966554,3.190356
18,0.500000,0.000000,0.500000,3.401692,2.997754,3.199723
20,0.500000,0.333333,0.166667,3.499880,3.016546,3.258213
24,0.666667,0.333333,0.000000,3.499243,3.027158,3.263201
15,0.333333,0.333333,0.333333,3.590039,3.097781,3.343910
